# 同步机制

第 1 章说过："搬运和计算是异步并行的，谁消费数据，谁就要等谁生产完"。本节讲 Pro 落实这句话的两套机制：**核内跨 Pipe 同步**（auto_mutex 自动 / 手动接口）与**跨核同步**（cross-core 事件）。


## 环境准备

本节代码面向 Ascend 950 环境。运行前先完成基础环境配置（详见 01.06）：设置设备号、关闭设备后端自动加载，并完成导入自检。

In [ ]:
import os

os.environ.setdefault('TILE_FWK_DEVICE_ID', '0')
os.environ['TORCH_DEVICE_BACKEND_AUTOLOAD'] = '0'

import pypto_pro.language as pl
import torch
import torch_npu

print('导入完成；设备编译、执行与精度仍需在 950 验证')

## 1. 同步问题全景

<table style="margin-left:0; margin-right:auto; text-align:left;">
<tr><th style="text-align:left;">场景</th><th style="text-align:left;">机制</th></tr>
<tr><td>同一执行域内，搬入→计算→搬出的跨 Pipe 依赖</td><td><b>auto_mutex</b>（TileGroup mutex 元数据，自动插入）</td></tr>
<tr><td>未配 mutex 的 Tile / 框架无法识别的依赖</td><td><b>手动同步接口</b>（核内，<code>pl.system.sync_src / sync_dst</code>）</td></tr>
<tr><td>Cube 执行域写、Vector 执行域读（或反向）</td><td><b>set/wait_cross_core</b>（跨核事件）</td></tr>
<tr><td>跨物理 Block / subblock 的依赖</td><td>cross-core 事件（sync_mode 选择）</td></tr>
</table>

判断口诀：**消费者读取本次 Kernel 中由另一个执行域（或另一个 Block）写入的数据时，需要跨核同步**；否则按核内依赖处理。


## 2. auto_mutex：自动同步

`@pl.jit(auto_mutex=True)` 时，框架根据 TileGroup 声明的 mutex_ids，为**能识别的数据依赖**自动插入同步：

```text
外层 Kernel：@pl.jit(auto_mutex=True)
  在 Kernel 内定义 TileType
  创建 src_group / out_group，分别配置 mutex_ids=[0] / [1]
  在 section_vector 中：load → 计算 → store
  编译器据 TileGroup 元数据为相关跨 Pipe 访问插入同步
```

三条边界：

1. **auto_mutex 只处理框架能通过 TileGroup 识别的依赖**——用 `make_tile` 创建的单块 Tile 没有 mutex 元数据，auto_mutex 无法保护它们；它不能替代所有显式同步；
2. 配置非空 mutex_ids 必须启用 auto_mutex，否则当前解析器会拒绝；只有 auto_mutex=True 却未配置 mutex 元数据，也不能自动保护这些 Tile 的跨 Pipe 依赖；
3. 若同时手动调用 `pl.system.mutex_lock/mutex_unlock`，其 ID 要与自动同步占用的 ID 分开，避免重复获取未释放的互斥资源。


## 3. 跨核同步：set/wait_cross_core

`pl.system.set_cross_core` 与 `pl.system.wait_cross_core` 保证**生产者与消费者之间的数据读写顺序**。事件只传递"可以继续执行"的信号，**不搬运数据、不分配共享缓冲**。

基本规则：**生产者在最后一个写操作之后 set；消费者在第一个读操作之前 wait**。

```python
with pl.section_cube():
    ...
    pl.store(workspace, acc_tile, [m_off, n_off])
    pl.system.set_cross_core(
        pipe=pl.PipeType.FIX,              # 同步点紧邻的数据操作所在流水
        event_id=0,
        sync_mode=pl.CrossCoreSyncMode.INTRA_BLOCK,
    )

with pl.section_vector():
    pl.system.wait_cross_core(
        pipe=pl.PipeType.MTE2,             # 与紧邻的 load 操作一致
        event_id=0,
        sync_mode=pl.CrossCoreSyncMode.INTRA_BLOCK,
    )
    pl.load(vec_tile, workspace, [m_off, n_off])
```

【待 950 验证：示例取自官方跨核同步文档（Cube 写 workspace → Vector 读）】

### 三个关键参数

- **pipe**：执行 set/wait 的硬件流水，**必须与同步点紧邻的数据操作一致**。常见组合：Cube 写 GM 后 Vector 读（`FIX → MTE2`）；Cube 结果搬 Vec 后 Vector 算（`FIX → V`）；Vector 写 Mat 后 Cube 读（`MTE3 → MTE1`）；
- **event_id**：本课程默认 `INTRA_BLOCK`，使用 [0,16) 的 ID，并让配对 set/wait 使用相同 ID 与模式；一个 ID 不应混用于尚未完成的无关依赖。`UNICAST_BLOCK` 的 AIC/AIV 编号映射另见 API，不能直接套用“同号”规则；
- **sync_mode**：参与者拓扑。AIC 与同 Block 内两个 AIV 子核协同用 `INTRA_BLOCK`（AIV→AIC 时两个 AIV 都须 set，AIC→AIV 时两个 AIV 都须 wait）；两个 AIV 子核之间用 `INTER_SUBBLOCK`；多个物理 Block 的同类核全核同步用 `INTER_BLOCK`，该模式不在 AIC 与 AIV 之间同步；单个 AIV 子核与 AIC 配对可用 `UNICAST_BLOCK`；其事件映射与双 AIV 会合语义不同，使用前需逐项核对。


## 4. 单向同步与双向同步

**单向同步**（一次性 workspace 或每轮独立地址）：只需生产者→消费者的 READY 事件。

```text
Cube/FIX（生产者）              Vector/MTE2（消费者）
写 workspace
set READY0 ──────────────────> wait READY0
                                读 workspace
```

**双向同步**（循环复用同一组缓冲）：必须同时建立正向与反向同步——

- 正向（READY）：保证消费者在数据**写完**后再读；
- 反向（RELEASE）：保证生产者在消费者**读完**后再覆盖该缓冲。

只建正向同步无法保护正在被消费的数据，可能导致后续写入覆盖尚未读完的内容。

> 设计粒度提醒：若 Cube 逐槽位生产、Vector 逐槽位消费，事件也要按相同粒度建立；把唯一的 set 放在生产循环之后，只能形成"整体阶段屏障"，无法实现逐 Tile 流水。


## 5. 设计流程小结

1. 先确定中间数据放哪（GM workspace / Mat / Vec）；
2. 确定生产者与消费者（谁写谁读、是否跨执行域/Block）；
3. 放置 set/wait（生产者末写后 set、消费者首读前 wait）；
4. 分配 event_id 并检查配对（同 ID 同 mode、不复用未消费事件）；
5. 循环复用缓冲时补齐反向事件。


### 以双输入加法列出四类依赖

MTE2 写输入后，V 才能读：这是本轮“先写后读”。V 读完输入后，下一轮 MTE2 才能覆盖同一个槽：这是跨轮“先读后写”。V 写输出后，MTE3 才能读走结果；MTE3 读完后，下一轮 V 才能覆盖输出。

因此双输入、单输出缓冲仍然可以正确，只是下一次结果写入可能等待前次 store 完成。若只把输入改成双缓冲，却忘记输出的复用依赖，仍可能产生错误。跨执行域交接时也沿同一思路列 READY 与 RELEASE，但需要使用对应的跨核事件，不能把核内 mutex ID 当成跨核 event_id。


## 6. 课后练习

1. （填空）跨核同步中，生产者应在 ______ 操作之后 set 事件，消费者应在 ______ 操作之前 wait 事件。
2. （选择）循环复用同一槽位缓冲时只建立正向同步，最可能的后果是：
   A. 编译失败<br/>
   B. 生产者覆盖尚未读完的数据<br/>
   C. 无影响
3. （判断）`event_id` 与 `mutex_id` 是同一种同步资源。（对/错）
4. （实践）设计一个场景：Vector 逐 Tile 产出中间结果写入 GM workspace，Cube 分批读取做矩阵乘。列出你的事件清单（方向、pipe、event_id、粒度）。

In [ ]:
!cat ./answer/02.06_answer.txt

## 7. 本节小结

- 两套机制：**核内跨 Pipe（auto_mutex / 手动）**、**跨核（cross-core 事件）**；
- auto_mutex 依赖 TileGroup 的 mutex 元数据，只覆盖可识别依赖；
- cross-core：生产者末写后 set、消费者首读前 wait；pipe 与紧邻数据操作一致；event_id [0,16) 成对使用；
- **循环复用缓冲必须双向同步**（READY + RELEASE）。

第 5 章融合算子与第 9 章协同编程会大量使用本节内容。